# 03 — Query Ingested Data & Compare Approaches

Verifies that Zerobus-pushed data arrived and runs the **same analytics**
as Part 1 for a direct comparison.

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

ZEROBUS_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_zerobus"

## Verify data arrived

In [ ]:
zb_df = spark.table(ZEROBUS_TABLE)
print(f"Zerobus table row count: {zb_df.count()}")
zb_df.show(10, truncate=False)

## Revenue by Casino Floor

Same query as Part 1, notebook 04 — but on the Zerobus-ingested table.

In [ ]:
spark.sql(f"""
    SELECT
        casino_floor,
        COUNT(*) AS total_plays,
        ROUND(SUM(bet_amount), 2) AS total_bets,
        ROUND(SUM(win_amount), 2) AS total_payouts,
        ROUND(SUM(bet_amount) - SUM(win_amount), 2) AS house_revenue
    FROM {ZEROBUS_TABLE}
    GROUP BY casino_floor
    ORDER BY house_revenue DESC
""").show(truncate=False)

## Top 10 Machines by Play Volume

In [ ]:
spark.sql(f"""
    SELECT
        machine_id,
        casino_floor,
        game_type,
        COUNT(*) AS total_plays,
        ROUND(SUM(bet_amount), 2) AS total_wagered
    FROM {ZEROBUS_TABLE}
    GROUP BY machine_id, casino_floor, game_type
    ORDER BY total_plays DESC
    LIMIT 10
""").show(truncate=False)

## Win/Loss Distribution by Game Type

In [ ]:
spark.sql(f"""
    SELECT
        game_type,
        COUNT(*) AS total_plays,
        SUM(CASE WHEN win_amount > 0 THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN win_amount = 0 THEN 1 ELSE 0 END) AS losses,
        ROUND(SUM(CASE WHEN win_amount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS win_pct,
        ROUND(AVG(bet_amount), 2) AS avg_bet,
        ROUND(AVG(CASE WHEN win_amount > 0 THEN win_amount END), 2) AS avg_win_payout
    FROM {ZEROBUS_TABLE}
    GROUP BY game_type
    ORDER BY total_plays DESC
""").show(truncate=False)

## Approach Comparison: Structured Streaming vs. Zerobus Ingest

| | **Structured Streaming (Part 1)** | **Zerobus Ingest (Part 2)** |
|---|---|---|
| **How data arrives** | Files land in cloud storage (or Kafka topic) | Pushed directly from source to Lakehouse |
| **Infrastructure needed** | File storage + Auto Loader (or Kafka broker) | Just the Zerobus endpoint — no message bus |
| **Latency** | Seconds to minutes (depends on trigger interval) | Sub-5 seconds end-to-end |
| **Processing model** | Full Spark — transforms, joins, aggregations in the stream | Ingestion only — transform after landing |
| **Best for** | Complex ETL pipelines, multi-step transforms | High-volume event push (IoT, telemetry, clickstream) |
| **Kafka relationship** | Reads from Kafka as a source | Replaces Kafka entirely for single-destination flows |

**Bottom line:** If your data needs complex in-flight transformation, Structured Streaming
is the right tool. If you're pushing events from devices/apps and your sole destination
is the Lakehouse, Zerobus removes an entire infrastructure layer.